# Khmer Whisper Fine-Tuning on Google Colab

Run these cells from top to bottom. Start with a small test first, then increase the sample counts after it works.

## 1. Check GPU

In Colab, choose **Runtime > Change runtime type > T4 GPU** or another GPU before running training.

In [ ]:
!nvidia-smi

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Install packages

In [ ]:
!pip install -q torch torchaudio transformers datasets evaluate jiwer accelerate tensorboard soundfile librosa

## 3. Get your training script

Option A: upload `finetune_whisper_khmer.py` from your computer.

After running the cell, click **Choose Files** and select your Python file.

In [ ]:
from google.colab import files
uploaded = files.upload()
print(uploaded.keys())

If you put this project on GitHub, you can use this instead of uploading manually:

```python
!git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
%cd YOUR_REPO
```

## 4. Optional: use Google Drive for output

Use this if you want the trained model files to stay saved after Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/whisper-tiny-khmer'
CACHE_DIR = '/content/hf_cache'
print('Output:', OUTPUT_DIR)

If you do not want to use Google Drive, run this cell instead:

In [ ]:
OUTPUT_DIR = '/content/whisper-tiny-khmer'
CACHE_DIR = '/content/hf_cache'
print('Output:', OUTPUT_DIR)

## 5. Small test run

This checks that everything works before doing a long training run.

In [ ]:
!python finetune_whisper_khmer.py \
  --output-dir "$OUTPUT_DIR" \
  --cache-dir "$CACHE_DIR" \
  --model-name openai/whisper-tiny \
  --use-fleurs-train \
  --skip-ddd \
  --max-train-samples 100 \
  --max-eval-samples 20 \
  --max-test-samples 20 \
  --num-train-epochs 1 \
  --per-device-train-batch-size 4 \
  --per-device-eval-batch-size 4 \
  --logging-steps 5 \
  --eval-steps 25 \
  --save-steps 25 \
  --fp16

## 6. Bigger training run

Run this only after the small test works. Increase `--max-train-samples` gradually.

In [ ]:
!python finetune_whisper_khmer.py \
  --output-dir "$OUTPUT_DIR" \
  --cache-dir "$CACHE_DIR" \
  --model-name openai/whisper-tiny \
  --use-fleurs-train \
  --skip-ddd \
  --max-train-samples 1000 \
  --max-eval-samples 200 \
  --max-test-samples 200 \
  --num-train-epochs 3 \
  --per-device-train-batch-size 8 \
  --per-device-eval-batch-size 8 \
  --logging-steps 25 \
  --eval-steps 200 \
  --save-steps 200 \
  --fp16

## 7. Download trained model

If you did not save to Google Drive, zip and download the model folder.

In [ ]:
!zip -r whisper-tiny-khmer.zip "$OUTPUT_DIR"
from google.colab import files
files.download('whisper-tiny-khmer.zip')

## 8. Optional: push to Hugging Face Hub

Run this only if you have a Hugging Face account and token.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!python finetune_whisper_khmer.py \
  --output-dir "$OUTPUT_DIR" \
  --cache-dir "$CACHE_DIR" \
  --model-name openai/whisper-tiny \
  --use-fleurs-train \
  --skip-ddd \
  --max-train-samples 1000 \
  --max-eval-samples 200 \
  --max-test-samples 200 \
  --num-train-epochs 3 \
  --push-to-hub \
  --hub-model-id YOUR_USERNAME/whisper-tiny-khmer \
  --fp16